# Standalone SGD neural-network experiment
Ten seeded networks with two ReLU hidden layers (32 and 16 units), trained for
exactly 100 epochs using minibatch SGD (learning rate 0.01, no momentum,
batch size 64) and mean absolute error (MAE) loss between class probabilities
and one-hot winner labels. No early stopping or validation split.

Train on elections through 2019 and evaluate on 2024.
Load training rows from train.csv and evaluate on the 2024 rows in test.csv. All model and
preprocessing code is defined here, with no imports from project modules.


In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score
from IPython.display import display

project_root = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "TEST_TRAIN" / "train.csv").is_file()
)
train = pd.read_csv(project_root / "TEST_TRAIN" / "train.csv")
test = pd.read_csv(project_root / "TEST_TRAIN" / "test.csv")

FEATURES = [
    "country/region", "previous_majority_proportion", "previous_winner",
    "Conservative", "Labour", "LD", "incumbent", "previous_con_share",
    "previous_lib_share", "previous_lab_share", "previous_natSW_share",
    "projected_con_share", "projected_lib_share", "projected_lab_share",
]
CATEGORICAL = ["country/region", "previous_winner", "incumbent"]
NUMERIC = [column for column in FEATURES if column not in CATEGORICAL]
SEEDS = (1,3,5,7,9,11,13,15,17,19)
EPOCHS = 100
BATCH_SIZE = 64
LEARNING_RATE = 0.01
TRAIN_THROUGH = 2019
EVALUATION_YEAR = 2024

years = pd.to_numeric(train["election"], errors="raise")
training_data = train.loc[years <= TRAIN_THROUGH].copy()
evaluation_years = pd.to_numeric(test["election"], errors="raise")
evaluation_data = test.loc[evaluation_years == EVALUATION_YEAR].dropna(subset=["winner"]).copy()
if training_data.empty or training_data["winner"].isna().any():
    raise ValueError("Training requires nonempty data with known winners.")
if evaluation_data.empty:
    raise ValueError("No evaluation rows with known winners.")
print(f"Training through {TRAIN_THROUGH}: {len(training_data)} rows")
print(f"Evaluation in {EVALUATION_YEAR}: {len(evaluation_data)} rows")


Training through 2019: 5705 rows
Evaluation in 2024: 632 rows


In [2]:
# Fit every preprocessing step on training rows only.
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("impute", SimpleImputer(strategy="median", keep_empty_features=True)),
        ("scale", StandardScaler()),
    ]), NUMERIC),
    ("categorical", Pipeline([
        ("impute", SimpleImputer(strategy="constant", fill_value="__MISSING__",
                                keep_empty_features=True)),
        ("encode", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), CATEGORICAL),
])

X_train_matrix = np.asarray(
    preprocessor.fit_transform(training_data[FEATURES]), dtype=np.float32
)
X_evaluation_matrix = np.asarray(
    preprocessor.transform(evaluation_data[FEATURES]), dtype=np.float32
)
if not np.isfinite(X_train_matrix).all() or not np.isfinite(X_evaluation_matrix).all():
    raise ValueError("Preprocessing produced non-finite features.")
label_encoder = LabelEncoder().fit(training_data["winner"])
X_train = torch.from_numpy(X_train_matrix)
X_evaluation = torch.from_numpy(X_evaluation_matrix)
y_train = torch.tensor(label_encoder.transform(training_data["winner"]), dtype=torch.long)
y_train_one_hot = nn.functional.one_hot(
    y_train, num_classes=len(label_encoder.classes_)
).to(dtype=torch.float32)
training_dataset = TensorDataset(X_train, y_train_one_hot)
print(f"Training tensor: {tuple(X_train.shape)}; evaluation tensor: {tuple(X_evaluation.shape)}")


Training tensor: (5705, 30); evaluation tensor: (632, 30)


In [3]:
class NeuralNetwork(nn.Module):
    def __init__(self, input_size, number_of_classes):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 32),
            nn.ReLU(),
            nn.Linear(32, 16),
            nn.ReLU(),
            nn.Linear(16, number_of_classes),
        )

    def forward(self, features):
        return self.layers(features)  # Raw logits; apply softmax for MAE and evaluation.


models = []
training_history = []
# Small CPU networks avoid unnecessary thread overhead.
torch.set_num_threads(1)
for seed in SEEDS:
    with torch.random.fork_rng(devices=[]):
        torch.manual_seed(seed)
        model = NeuralNetwork(X_train.shape[1], len(label_encoder.classes_))
        optimizer = torch.optim.SGD(model.parameters(), lr=LEARNING_RATE, momentum=0.0)
        criterion = nn.L1Loss()
        loader = DataLoader(
            training_dataset, batch_size=BATCH_SIZE, shuffle=True,
            generator=torch.Generator().manual_seed(seed), num_workers=0,
        )
        for epoch in range(1, EPOCHS + 1):
            model.train()
            total_loss = 0.0
            for features, targets in loader:
                optimizer.zero_grad()
                probabilities = torch.softmax(model(features), dim=1)
                loss = criterion(probabilities, targets)
                if not torch.isfinite(loss):
                    raise RuntimeError(f"Non-finite loss: seed {seed}, epoch {epoch}")
                loss.backward()
                optimizer.step()
                total_loss += loss.item() * len(targets)
            training_history.append({
                "seed": seed, "epoch": epoch,
                "training_loss": total_loss / len(training_dataset),
            })
        model.eval()
        models.append(model)
        print(f"Seed {seed}: completed {EPOCHS} epochs; training loss {training_history[-1]['training_loss']:.4f}")
training_history = pd.DataFrame(training_history)


Seed 1: completed 100 epochs; training loss 0.0583
Seed 3: completed 100 epochs; training loss 0.0582
Seed 5: completed 100 epochs; training loss 0.0591
Seed 7: completed 100 epochs; training loss 0.0588
Seed 9: completed 100 epochs; training loss 0.0588
Seed 11: completed 100 epochs; training loss 0.0590
Seed 13: completed 100 epochs; training loss 0.0584
Seed 15: completed 100 epochs; training loss 0.0587
Seed 17: completed 100 epochs; training loss 0.0589
Seed 19: completed 100 epochs; training loss 0.0579


In [4]:
# Evaluate only after all ten fixed-duration fits have finished.
if len(models) != len(SEEDS):
    raise ValueError("Train all ten models before evaluation.")
probabilities_by_model = []
individual_scores = []
with torch.inference_mode():
    for seed, model in zip(SEEDS, models):
        model.eval()
        probabilities = torch.softmax(model(X_evaluation), dim=1).numpy()
        probabilities_by_model.append(probabilities)
        predictions = label_encoder.inverse_transform(probabilities.argmax(axis=1))
        individual_scores.append({
            "seed": seed,
            "accuracy": accuracy_score(evaluation_data["winner"], predictions),
        })
display(pd.DataFrame(individual_scores))

ensemble_probabilities = np.mean(probabilities_by_model, axis=0)
ensemble_predictions = label_encoder.inverse_transform(ensemble_probabilities.argmax(axis=1))
ensemble_accuracy = accuracy_score(evaluation_data["winner"], ensemble_predictions)
print(f"{EVALUATION_YEAR} ensemble accuracy: {ensemble_accuracy:.2%} ({len(evaluation_data)} rows)")
complete = evaluation_data[FEATURES].notna().all(axis=1)
if complete.any():
    complete_accuracy = accuracy_score(
        evaluation_data.loc[complete, "winner"], ensemble_predictions[complete.to_numpy()]
    )
    print(f"Complete-row accuracy: {complete_accuracy:.2%} ({complete.sum()} rows)")


,seed,accuracy
0,1,0.757911
1,3,0.772152
2,5,0.745253
3,7,0.750000
4,9,0.795886
5,11,0.778481
6,13,0.756329
7,15,0.794304
8,17,0.748418
9,19,0.786392


2024 ensemble accuracy: 77.22% (632 rows)
Complete-row accuracy: 77.62% (621 rows)
